# 01 — Kvantitative bøyningsprofiler og stabilitet over tid (1950–2020)

Denne notebooken kobler **Norsk Ordbank (`sqlite/ordbank.db`)** med **Nasjonalbibliotekets historiske n-grammer (`sqlite/ngram.py`)** for å undersøke de to første hovedspørsmålene i prosjektet:

1. **Hvor stor andel av et paradigme kan én enkelt bøyningsform få — og hva er «normalen»?**
   - Hvilke verb har *infinitiv* på topp (f.eks. *vite*, *spise*, *forstå*), hvilke har *presens* på topp (f.eks. *ha*, *gjøre*), og hvilke har *preteritum* på topp (f.eks. *gå $\rightarrow$ gikk*, *komme $\rightarrow$ kom*, *si $\rightarrow$ sa*)?
   - Hvilke substantiv domineres av *ubestemt entall* (*eple*, *bil*, *år*), *bestemt entall* (*bussen*, *sola/solen*, *krigen*) eller *flertall*?
2. **Hvor stabile er disse bøyningsprofilene over tid (1950–2020)?**
   - Holder den dominerende bøyningsformen seg på topp gjennom hele perioden 1950–2020?
   - Hvor lite varierer proporsjonene fra år til år og fra tiår til tiår?

In [ ]:
import sys
from pathlib import Path

# Sørg for at prosjektroten ligger i sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sqlite.queries import (
    lookup_wordform,
    get_paradigm,
    get_verb_profile_forms,
    get_noun_profile_forms,
    find_anchor_forms,
    VERB_TENSE_SLOTS,
    VERB_CORE_SLOTS,
    NOUN_CORE_SLOTS,
)
from sqlite.ngram import (
    fetch_ngram_dhlab,
    fetch_ngram_sp,
    compute_paradigm_profile,
)

## 1. Sjekk bøyningsparadigmer og flertydighet i Norsk Ordbank
La oss først se på paradigmene og eventuelle homografer for nøkkelordene fra abstractet.

In [ ]:
for verb in ["gå", "vite", "spise", "ta", "fiske"]:
    former = get_verb_profile_forms(grunnform=verb, slots=VERB_CORE_SLOTS)
    beskrivelse = ", ".join(
        f"{r['boy_tekst']}={r['oppslag']} (lem={r['antall_lemma_id']})" for r in former
    )
    print(f"{verb:8s}: {beskrivelse}")

In [ ]:
# Dekomponering av en flertydig ordform ('fisker') vs. entydige holdepunkter
for r in lookup_wordform("fisker"):
    anchors = find_anchor_forms(r["lemma_id"])["helt_entydige"]
    anchor_str = ", ".join(f"{a['boy_tekst']}={a['oppslag']}" for a in anchors)
    print(f"lemma_id={r['lemma_id']:6d} | {r['grunnform']:8s} ({r['pos']:5s}, {r['boy_tekst']:6s}) -> Entydige holdepunkter: {anchor_str}")

## 2. Verbprofiler 1950–2020: Infinitiv vs. presens vs. preteritum
Alle hentede n-grammer mellomlagres automatisk i `sqlite/ngram_cache.db`, slik at gjentatte kjøringer skjer på millisekunder.

In [ ]:
from sqlite.analyze_profiles import run_analysis

run_analysis(period=(1950, 2020))

## 3. Tiårsstabilitet (1950-tallet til 2010-tallet) for utvalgte verb og substantiv
Her ser vi hvordan proporsjonene mellom bøyningsformene utvikler seg tiår for tiår.

In [ ]:
def vis_tiaarsprofil(grunnform: str, pos: str = "verb"):
    p = compute_paradigm_profile(grunnform=grunnform, pos=pos, period=(1950, 2020))
    slots = list(p["overall_proportions"].keys())
    print(f"\n=== {grunnform.upper()} ({pos}, lemma_id={p['lemma_id']}, totalt={p['grand_total']:,} treff) ===")
    print(f"Dominerende kategori: {p['dominant_slot']} ({p['dominant_share']*100:.1f}%), på topp i {p['top_slot_consistency']*100:.1f}% av årene")
    header = f"{'Tiår':12s} | " + " | ".join(f"{s:8s}" for s in slots)
    print(header)
    print("-" * len(header))
    for dec, vals in p["decade_profiles"].items():
        row_str = " | ".join(f"{vals.get(s, 0.0)*100:7.1f}%" for s in slots)
        print(f"{dec:12s} | {row_str}")

for v in ["gå", "vite", "spise", "ta", "komme"]:
    vis_tiaarsprofil(v, pos="verb")

for n in ["eple", "buss", "fisk", "fisker"]:
    vis_tiaarsprofil(n, pos="subst")